In [2]:
import re
innotes = 0
roman_to_arabic = {
    'I': 1, 'II': 2, 'III': 3, 'IV': 4, 'V': 5,
    'VI': 6, 'VII': 7, 'VIII': 8, 'IX': 9, 'X': 10,
    'XI': 11, 'XII': 12, 'XIII': 13, 'XIV': 14, 'XV': 15,
    'XVI': 16, 'XVII': 17, 'XVIII': 18, 'XIX': 19, 'XX': 20,
    'XXI': 21, 'XXII': 22, 'XXIII': 23, 'XXIV': 24, 'XXV': 25,
    'XXVI': 26, 'XXVII': 27, 'XXVIII': 28, 'XXIX': 29, 'XXX': 30,
    'XXXI': 31, 'XXXII': 32, 'XXXIII': 33, 'XXXIV': 34, 'XXXV': 35,
    'XXXVI': 36, 'XXXVII': 37, 'XXXVIII': 38, 'XXXIX': 39, 'XL': 40,
    'XLI': 41, 'XLII': 42, 'XLIII': 43, 'XLIV': 44, 'XLV': 45,
    'XLVI': 46, 'XLVII': 47, 'XLVIII': 48, 'XLIX': 49, 'L': 50
}


fname = '/Users/gcrane/github/GRC_misc/tlg0548.tlg002.wagner1894-grc0.xml'
fname = '/Users/gcrane/github/GRC_misc/tlg0548.tlg001.wagner1894-grc0.xml'
f = open(fname)



outfname = '/Users/gcrane/github/GRC_misc/tlg0548.tlg002.wagner1894-grc1.xml'
outfname = '/Users/gcrane/github/GRC_misc/tlg0548.tlg001.wagner1894-grc1.xml'
tmpfname = '/tmp/tmp.xml'
outf = open(tmpfname,'w')

curpage = ''
innotes = 1
cursectn = 0
cursubsectn = 0
sentend = 0
prevaltsect = 0
curaltsect = 0
newsection = '</p></div></div>\n\n<div type="textpart" subtype="section" n="ZZZ">\n<div type="textpart" subtype="subsection" n="1"><p>'
newsubsection = '</p></div>\n\n<div type="textpart" subtype="subsection" n="ZZZ"><p>'
for l in f:
    if(re.search('"book"',l)):
        prevaltsect = 1
        curaltsect = 1
        
    m = re.search('"section" n="([0-9]+)"', l)
    if(m):
        cursectn = int(m[1])

    m = re.search('"subsection" n="([0-9]+)"', l)
    if(m):
        cursubsectn = int(m[1])
    l = re.sub('\s+$','',l)
    l = re.sub('^\s+','',l)
    if(re.search('<text',l)):
        innotes = 0
    m = re.search('<pb n="([0-9]+)',l)
    if(m):
        if(not innotes):
            print('nonotes',m[1])
        innotes = 0
        curpage = m[1]
        curline =0
        print('\n'+l,file=outf)
        continue

    if(re.search('(<div|<head|^<mile|<text|<body|div>|text>|body>)',l)):
        print(l,file=outf)
        continue


    if(re.search('<note>',l)):
        innotes = 1

    if(innotes):
        l = re.sub('([0-9])\s+(ss\.)','\g<1>\g<2>',l)
        l = re.sub('(\s+|^)([1-9]|1[0-9]|2[0-8])([s]*)\.\s+','</note>\n\n<note n="'+curpage+'.\g<2>\g<3>">',l)
        print(l,file=outf,end=' ')
        continue

        
    #if(re.search('[02468]$',curpage)):
    if(re.search('[13579]$',curpage)):
        if(not innotes):
            if(l and not re.search('<pb',l)):
                
                curline = curline + 1
                l = '<lb n="'+curpage+'.'+str(curline)+'"/>' + l
                if(curline in [5,10,15,20,25]):
                    m = re.search('([0-9]+$)',l)
                    if(not m):
                        if(curline > 19 and 0):
                            print('nolnum',curpage,curline,l)
                    else:
                        lnum = int(m[1])
                        if(not lnum == curline):
                            if(curline > 19 and 0):
                                print('mism',curpage,curline,l)
                        else:
                            l = re.sub('\s*([0-9]+$)','',l)
            m = re.search('^([0-9]+)',l)
            if(m):
                if(re.search('[^A-Zz-z]\.\s',l) and 0):
                    l = re.sub('([^A-Zz-z]\.)\s(.+)\s+([0-9]+)','\g<1></p></div>\n\n<div type="textpart" subtype="section" n="\g<3>">\n<p>\g<2>',l)
                    print(l)
            m = re.search('(<lb[^>]+>)([0-9]+)',l)
            if(m):
                workn = int(m[2])
                #print('a','workn',workn,'sub',cursubsectn)
                if(not workn == cursubsectn + 1 and not workn == cursectn + 1):
                    print(curpage,'jump',workn,cursectn,cursubsectn)
                if(not workn == cursubsectn + 1):
                    cursectn = workn
                    cursubsectn = 1
                    #print('nowa',cursectn,cursubsectn)
                    substr = re.sub('ZZZ',str(workn),newsection)
                else:
                    cursubsectn = workn
                    substr = re.sub('ZZZ',str(workn),newsubsection)
                if(sentend):
                    l = substr + l
                elif(re.search('\.\s+',l)):
                    l = re.sub('(\.)\s+','\g<1>'+substr,l)
                elif(re.search('·\s+',l)):
                    l = re.sub('(·)\s+','\g<1>'+substr,l)
                elif(re.search('·\s+',l)):
                    l = re.sub('(·)\s+','\g<1>'+substr,l)
                elif(re.search('@',l)):
                    l = re.sub('\s*@\s*',substr,l)
                else:
                    l = substr +'NOB'+l
                l = re.sub('(<lb[^>]+>)\s*([0-9]+)\s*','\g<1>',l)
                    
                #print(curpage,workn,'sect',cursectn,'sub',cursubsectn,l)
    else:
        if(not innotes and 1):
            if(l and not re.search('<pb',l)):
                
                curline = curline + 1
                if(curline in [5,10,15,20,25]):
                    m = re.search('^([0-9]*)(5|[123][05])\s+',l)
                    if(not m):
                        if(curline > 19 and 0):
                            print('nolnum',curpage,curline,l)
                    else:
                        #print('m1',m[1]+m[2],l)
                        lnum = int(m[1]+m[2])
                        if(not lnum == curline):
                            if(curline > 19 and 0):
                                print('mism',curpage,curline,l)
                        else:
                            
                           l = re.sub('^([0-9]*)(5|[123][05])\s+','',l) 
                l = '<lb n="'+curpage+'.'+str(curline)+'"/>' + l
            m = re.search('([0-9]+)$',l)
            if(m and 0):
                if(re.search('[^A-Zz-z]\.\s',l) and 0):
                    l = re.sub('([^A-Zz-z]\.)\s(.+)\s+([0-9]+)','\g<1></p></div>\n\n<div type="textpart" subtype="section" n="\g<3>">\n<p>\g<2>',l)
                    print(l)
            m = re.search('([0-9]+)$',l)
            if(m):
                workn = int(m[1])
                #print('b','workn',workn,'sub',cursubsectn)
                if(not workn == cursubsectn + 1):
                    cursectn = workn
                    cursubsectn = 1
                    #print('nowb',cursectn,cursubsectn)
                    substr = re.sub('ZZZ',str(workn),newsection)
                else:
                    cursubsectn = workn
                    substr = re.sub('ZZZ',str(workn),newsubsection)
                if(sentend):
                    l = substr + l
                elif(re.search('\.\s+',l)):
                    l = re.sub('(\.)\s+','\g<1>'+substr,l)
                elif(re.search('·\s+',l)):
                    l = re.sub('(·)\s+','\g<1>'+substr,l)
                elif(re.search('·\s+',l)):
                    l = re.sub('(·)\s+','\g<1>'+substr,l)
                elif(re.search('@',l)):
                    l = re.sub('\s*@\s*',substr,l)
                else:
                    l = substr +'NOB'+l
                #print(curpage,workn,'sect',cursectn,'sub',cursubsectn,l)
                l = re.sub('\s*([0-9]+)$','',l)

    if(l):
        if(re.search('[\.;][\s0-9]*$',l) or re.search('·$',l)):
            sentend = 1
        else:
            sentend = 0
    print(l,file=outf)
f.close()
outf.close()


f = open(tmpfname)
outfname = '/Users/gcrane/github/GRC_misc/tlg0548.tlg001.wagner1894-grc1.xml'
outf = open(outfname,'w')
innotes = 0
curbook = ''
for l in f:
    l = re.sub('\s+$','',l)
    l = re.sub('^\s+','',l)
    m = re.search('"book" n="([0-9]+)"', l)
    if(m):
        curbook = m[1]

    m = re.search('<pb n="([^"]+)',l)
    if(m):
        curpage = m[1]
        innotes = 0

    if(re.search('<note',l)):
        innotes = 1
    if(innotes):
        print(l,file=outf)
        continue

    if(re.search('[13579]$',curpage)):
        m = re.search('\s+([0-9]+)$',l)
        if(m):
            curaltcit = '<milestone unit="altsect" n="'+curbook+'.'+m[1]+'"/>'
            l = re.sub('\s+([0-9]+)$','',l)
            if(re.search('</p></div></div>',l)):
                l = re.sub('</p></div></div>','</p></div></div>'+curaltcit,l)
            elif(re.search('</p></div>',l)):
                l = re.sub('</p></div>','</p></div>'+curaltcit,l)
            elif(re.search('<p>',l)):
                l = re.sub('(<p>)','\g<1>'+curaltcit,l)
            elif(re.search('\.\s+',l)):
                l = re.sub('(\.)\s+','\g<1>'+curaltcit+' ',l)
            elif(re.search('(<lb[^>]+>)',l)):
                l = re.sub('(<lb[^>]+>)\s*','\g<1>'+curaltcit+' ',l)
            else:
                print('noalt',l)

    else:
        m = re.search('(<lb[^>]+>|<p>)([0-9]+)',l)
        if(m):
            l = re.sub('(<lb[^>]+>|<p>)([0-9]+)','\g<1>',l)
            curaltcit = '<milestone unit="altsect" n="'+curbook+'.'+m[2]+'"/>'
            if(re.search('</p></div></div>',l)):
                l = re.sub('</p></div></div>','</p></div></div>'+curaltcit,l)
            elif(re.search('</p></div>',l)):
                l = re.sub('</p></div>','</p></div>'+curaltcit,l)
            elif(re.search('<p>',l)):
                l = re.sub('(<p>)','\g<1>'+curaltcit,l)
            elif(re.search('\.\s+',l)):
                l = re.sub('(\.)\s+','\g<1>'+curaltcit+' ',l)
            elif(re.search('(<lb[^>]+>)',l)):
                l = re.sub('(<lb[^>]+>)\s*','\g<1>'+curaltcit+' ',l)
            else:
                print('noalt',l)
        
        
    if(l):
        if(re.search('[\.;][\s0-9]*$',l) or re.search('·$',l)):
            sentend = 1
        else:
            sentend = 0


    print(l,file=outf)

outf.close()
f.close()

f = open(outfname)

prevsect = 0
cursect = 0
for l in f:
    m = re.search('<milestone unit="altsect" n="[0-9]\.([^"]+)',l)
    if(m):
        cursect = int(m[1])
        if(cursect == 1):
            prevsect = 1
            continue
        if(not cursect == prevsect + 1):
            print('altjump',prevsect,cursect,l)
        prevsect = cursect

f.close()
        
        
        

nonotes 5
